# Statistics reported in the paper

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

for _cand in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_cand / "layer3_reliability.py").exists():
        sys.path.insert(0, str(_cand))
        break
import layer3_reliability as rel
import crossfarm_nested as cn

OUT = cn.OUT
OUT_DIR = OUT / "18n_reported_statistics"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TARGETS = ["is_nonvocal", "is_distress_related", "is_resource_related", "proxy_functional_v1"]
COVERAGES = [1.0, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]
SOURCES = {
    "discrete_units": (OUT / "14n_layer2_crossfarm_nested" / "crossfarm_fold_predictions.csv.gz",
                       OUT / "19n_layer3_reliability_analysis" / "framework_inner_predictions.csv.gz",
                       "target_name", "heldout_farm"),
    "continuous_embeddings": (OUT / "14c_layer2_continuous_embeddings" / "continuous_fold_predictions.csv.gz",
                              OUT / "14c_layer2_continuous_embeddings" / "continuous_inner_predictions.csv.gz",
                              "target_name", "heldout_farm"),
    "cnn_logmel": (OUT / "cnn_baseline" / "cnn_lofo_predictions.csv.gz",
                   OUT / "cnn_baseline" / "cnn_lofo_inner_predictions.csv.gz", "target", "held_farm"),
}
outer, inner = {}, {}
for m, (po, pi, tcol, hcol) in SOURCES.items():
    outer[m] = rel.standardise_predictions(pd.read_csv(po), tcol, hcol)
    inner[m] = rel.standardise_predictions(pd.read_csv(pi), tcol, hcol, "inner_val_farm")

In [ ]:
param_rows, f1_rows = [], []
for m in SOURCES:
    for t in TARGETS:
        o, i = outer[m][outer[m].target == t], inner[m][inner[m].target == t]
        task = o["task"].iloc[0]
        for held in sorted(o["held"].unique()):
            oo, ii = o[o.held == held].reset_index(drop=True), i[i.held == held].reset_index(drop=True)
            Po, classes = rel.prediction_matrix(oo)
            Pi, _ = rel.prediction_matrix(ii)
            res = rel.nested_fold_reliability({"P": Pi, "y": ii["y"].to_numpy(), "farms": ii["inner_val_farm"].to_numpy()},
                                              {"P": Po}, task, len(classes))
            temp = np.nan
            if res["method"] == "temperature":
                temp = (rel._fit_temperature_binary(Pi[:, 1], ii["y"].to_numpy()) if task == "binary"
                        else rel._fit_temperature_multiclass(Pi, ii["y"].to_numpy()))
            param_rows.append({"model": m, "target": t, "heldout_farm": held, "method": res["method"],
                               "score": res["score"], "temperature": temp, "learned_threshold": res["threshold"]})
        r = rel.nested_reliability(outer[m], inner[m], t)
        y = r["y"]
        f1_rows.append({"model": m, "target": t,
                        **{f"macro_f1_{k}": f1_score(y, r[f"{k}_P"].argmax(1), average="macro", zero_division=0)
                           for k in ("raw", "cal", "sel")}})
params_df = pd.DataFrame(param_rows)
f1_df = pd.DataFrame(f1_rows)
display(params_df.round(3))
display(f1_df.round(3))

In [ ]:
units = pd.read_csv(SOURCES["discrete_units"][0])
bin_rows = []
for t in TARGETS:
    s = units[units.target_name == t]
    if s.task_type.iloc[0] == "binary":
        conf = np.maximum(s.proba__1, 1 - s.proba__1).to_numpy()
    else:
        conf = s[[c for c in s.columns if c.startswith("proba__") and c != "proba__1"]].max(1).to_numpy()
    ok = (s.y_pred == s.y_true).to_numpy()
    q = pd.qcut(conf, 5, labels=False, duplicates="drop")
    for b in sorted(set(q)):
        bin_rows.append({"target": t, "bin": int(b), "n_clips": int((q == b).sum()),
                         "mean_confidence": float(conf[q == b].mean()), "accuracy": float(ok[q == b].mean())})
bins_df = pd.DataFrame(bin_rows)
display(bins_df.round(3))

In [ ]:
f = units[units.target_name == "proxy_functional_v1"]
class_rows = []
for cls, g in f.groupby("y_true_label"):
    row = {"true_class": cls, "n_clips": len(g), "n_correct": int((g.y_pred_label == cls).sum())}
    for farm, gf in g.groupby("heldout_farm"):
        row[f"farm{farm}_clips"] = len(gf)
        row[f"farm{farm}_errors"] = int((gf.y_pred_label != cls).sum())
    class_rows.append(row)
class_df = pd.DataFrame(class_rows)
display(class_df)

man = pd.read_csv(OUT / "00_manifest" / "manifest.csv").set_index("clip_id")
d = units[(units.target_name == "is_distress_related") & (units.y_true == 0)].sort_values("proba__1", ascending=False).head(12)
top_distress = d[["_record_key", "heldout_farm", "proba__1", "bundle_name"]].assign(
    barn_zone=d._record_key.map(man.barn_zone), main_category=d._record_key.map(man.main_category_final))
display(top_distress)

clips = cn.load_clip_table()
known = clips[clips.farm_num.isin(cn.FARMS)]
counts = pd.concat([known.groupby("farm_num")[["is_nonvocal", "is_distress_related", "is_resource_related"]].sum(),
                    known.groupby("farm_num").size().rename("n_clips"),
                    pd.crosstab(known.farm_num, known.proxy_functional_v1)], axis=1)
display(counts)

In [ ]:
from sklearn.metrics import roc_auc_score
auc_rows = []
for m in SOURCES:
    for t in TARGETS[:3]:
        o = outer[m][outer[m].target == t]
        for farm, g in list(o.groupby("held")) + [("pooled", o)]:
            y, p1 = g["y"].to_numpy(), g["proba__1"].to_numpy()
            auc_rows.append({"model": m, "target": t, "heldout_farm": farm, "n_clips": len(g),
                             "n_positive": int(y.sum()),
                             "auroc": float(roc_auc_score(y, p1)) if 0 < y.sum() < len(y) else np.nan})
auc_df = pd.DataFrame(auc_rows)
display(auc_df.pivot_table(index=["target", "heldout_farm"], columns="model", values="auroc").round(3))

In [ ]:
params_df.to_csv(OUT_DIR / "nested_calibration_parameters.csv", index=False)
f1_df.to_csv(OUT_DIR / "macro_f1_raw_calibrated_selected.csv", index=False)
bins_df.to_csv(OUT_DIR / "confidence_bin_accuracy.csv", index=False)
class_df.to_csv(OUT_DIR / "functional_class_errors.csv", index=False)
top_distress.to_csv(OUT_DIR / "top_non_distress_by_distress_probability.csv", index=False)
counts.to_csv(OUT_DIR / "class_counts_by_farm.csv")
auc_df.to_csv(OUT_DIR / "auroc_by_farm.csv", index=False)